# `sovereign_rating(country=None, on=None, revision=None)`

The rating behind each country's spread and premiums, from `macro.damodaran_country_risk`. `basis` says how he rated it: `rating` (Moody's, Aaa best to C worst) or `prs` (no Moody's rating, so the PRS Group's composite score, 0 to 100, higher safer, stands in).

Every cell runs against prod, read-only, as `readonly_user`. Change the arguments and re-run any cell.

`country` is an ISO alpha-3 code (`AE-AZ`, `AE-RK`, `AE-SH` for the three emirates) or a name from that edition, in any case and spacing; `None` gives every country. `on` picks the edition in force on that date (default: today in US/Eastern), and `revision` defaults to its newest. Values are Damodaran's, as published and never recomputed, in fractions: 0.0442 is 4.42%.

In [1]:
from datetime import date

import polars as pl
from IPython.display import Markdown

from arora_data.sovereign_rating import SovereignRating
from arora_data.sync import sovereign_rating
from arora_data.dates import today_et
from arora_data.frames import to_df


def show(rows):
    """One result or a list of results as a Markdown table."""
    rows = [r.model_dump() for r in (rows if isinstance(rows, list) else [rows])]
    if not rows:
        return Markdown("0 rows")
    head = list(rows[0])
    lines = ["|".join(head), "|".join("---" for _ in head)]
    lines += ["|".join(str(r[c]).replace("|", "/") for c in head) for r in rows]
    return Markdown(f"{len(rows)} rows\n\n" + "\n".join(lines))

## One country, by code or name

In [2]:
assert sovereign_rating("JPN") == sovereign_rating("japan")
show(sovereign_rating("JPN"))

1 rows

edition_date|revision|entity_code|published_name|entity_type|region|basis|moodys_rating|prs_score
---|---|---|---|---|---|---|---|---
2026-07-01|1|JPN|Japan|country|Asia|rating|A1|None

## How many by basis and rating

In [3]:
rating = to_df(sovereign_rating(), SovereignRating)
print(rating.group_by("basis").len())
rating.filter(pl.col("basis") == "rating").group_by("moodys_rating").len().sort("len", descending=True)

shape: (2, 2)
┌────────┬─────┐
│ basis  ┆ len │
│ ---    ┆ --- │
│ str    ┆ u32 │
╞════════╪═════╡
│ rating ┆ 157 │
│ prs    ┆ 21  │
└────────┴─────┘


moodys_rating,len
str,u32
"""Caa1""",14
"""Aaa""",12
"""Baa3""",12
"""Aa3""",11
"""Ba3""",11
…,…
"""Aa2""",4
"""Baa2""",4
"""Aa1""",3


## Countries scored by PRS
No Moody's rating, so no `moodys_rating`; the score sets the spread.

In [4]:
rating.filter(pl.col("basis") == "prs").sort("prs_score").select("entity_code", "published_name", "prs_score")

entity_code,published_name,prs_score
str,str,"decimal[38,2]"
"""SDN""","""Sudan""",46.00
"""YEM""","""Yemen, Republic""",50.75
"""PRK""","""Korea, D.P.R.""",52.50
"""MWI""","""Malawi""",54.25
"""MMR""","""Myanmar""",55.00
…,…,…
"""GMB""","""Gambia""",67.00
"""LBY""","""Libya""",68.00
"""RUS""","""Russia""",70.50


## Rating changes between April and July 2026

In [5]:
april = to_df(sovereign_rating(on=date(2026, 4, 1)), SovereignRating)
july = to_df(sovereign_rating(on=date(2026, 7, 1)), SovereignRating)
(april.join(july, on="entity_code", suffix="_july")
 .filter(pl.col("moodys_rating") != pl.col("moodys_rating_july"))
 .select("entity_code", "published_name", "moodys_rating", "moodys_rating_july"))

entity_code,published_name,moodys_rating,moodys_rating_july
str,str,str,str
"""BEL""","""Belgium""","""Aa3""","""A1"""
"""BHS""","""Bahamas""","""B1""","""Ba3"""
"""BMU""","""Bermuda""","""A2""","""A1"""
"""BRB""","""Barbados""","""B2""","""C"""
"""COK""","""Cook Islands""","""B1""","""Ba3"""
…,…,…,…
"""MNE""","""Montenegro""","""B1""","""Ba3"""
"""MOZ""","""Mozambique""","""Caa3""","""Caa2"""
"""PRY""","""Paraguay""","""Baa3""","""Ba1"""


## Where Moody's withdrew a rating
His file maps S&P's rating onto Moody's scale: Venezuela's Moody's is withdrawn in 2026-07-01, so S&P's CCC- gives Caa3.

In [6]:
show(sovereign_rating("VEN", on=date(2026, 4, 1)) + sovereign_rating("VEN", on=date(2026, 7, 1)))

2 rows

edition_date|revision|entity_code|published_name|entity_type|region|basis|moodys_rating|prs_score
---|---|---|---|---|---|---|---|---
2026-04-01|1|VEN|Venezuela|country|Central and South America|rating|C|None
2026-07-01|1|VEN|Venezuela|country|Central and South America|rating|Caa3|None

## Known source issue
In the 2026-07-01 edition Barbados's Moody's rating is typed as C in his file (B2 in April), next to Belarus's C. Every Barbados value in that edition follows from it.

In [7]:
show(sovereign_rating("BRB", on=date(2026, 4, 1)) + sovereign_rating("BRB", on=date(2026, 7, 1)))

2 rows

edition_date|revision|entity_code|published_name|entity_type|region|basis|moodys_rating|prs_score
---|---|---|---|---|---|---|---|---
2026-04-01|1|BRB|Barbados|country|Caribbean|rating|B2|None
2026-07-01|1|BRB|Barbados|country|Caribbean|rating|C|None

## Bad input

In [8]:
for kwargs in [dict(country="South Korea"), dict(country="US"), dict(on=date(2014, 12, 31)),
               dict(on=date(2026, 3, 31), revision=7)]:
    try:
        sovereign_rating(**kwargs)
    except ValueError as e:
        print(e)

'South Korea' is not a stored country in Damodaran's 2026-07-01 edition: use an ISO alpha-3 code or a name from that edition, such as USA or 'United States'


'US' is not a stored country in Damodaran's 2026-07-01 edition: use an ISO alpha-3 code or a name from that edition, such as USA or 'United States'


no Damodaran edition on or before 2014-12-31: the first is 2015-01-01


Damodaran's 2026-01-01 edition has no revision 7: it has 1, 2, 3, 4
